In [57]:
import numpy as np
from scipy.io import wavfile
from scipy.signal import get_window
import scipy.fftpack as fft 

In [58]:
AUDIO_PATH = './sample[1].wav'
sr, audio = wavfile.read(AUDIO_PATH)

# print(sr)
# print(audio)
print(audio.shape)

(76032, 2)


In [59]:
# Konfigurasi Stereo ke Mono
if audio.ndim > 1:
    audio = np.mean(audio, axis=1)

In [60]:
print(sr)
print(audio)
print(audio.shape)
print(audio.ndim)

24000
[0. 0. 0. ... 0. 0. 0.]
(76032,)
1


In [61]:
FFT_SIZE = 2048
HOP_SIZE = 15
pad_width = int(FFT_SIZE / 2)

print(np.max(audio))
# Normalization
audio = audio / np.max(np.abs(audio))
print(np.max(audio))
# print(audio)

29877.0
1.0


In [62]:
# Frame Blocking
audio = np.pad(audio, pad_width, mode='reflect')
HOP_LENGTH = int(np.round(sr * HOP_SIZE / 1000))
FRAME_NUM = int(((len(audio) - FFT_SIZE) // HOP_LENGTH) + 1)

In [64]:
print("Hop Length: ", HOP_LENGTH)
print("Frame Number: ", FRAME_NUM)

Hop Length:  360
Frame Number:  212


In [65]:
frames = np.zeros((FRAME_NUM, FFT_SIZE))

for i in range(FRAME_NUM):
    start = i * HOP_LENGTH
    end = start + FFT_SIZE
    frames[i] = audio[start:end]

In [67]:
print("Frames: ", frames)

Frames:  [[ 3.34705626e-05 -3.34705626e-05  0.00000000e+00 ...  3.34705626e-05
   0.00000000e+00 -3.34705626e-05]
 [ 3.34705626e-05  1.00411688e-04  1.33882251e-04 ... -2.47682164e-03
  -5.02058440e-04 -1.40576363e-03]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -3.41399739e-03
  -6.49328915e-03 -1.07105800e-03]
 ...
 [ 3.34705626e-05  1.67352813e-05  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  3.34705626e-05 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]]


In [71]:
# Windowing
window_type = 'hann'

window = get_window(window_type, FFT_SIZE, fftbins=True)
audio_win = frames * window
print("Windowed Audio: ", audio_win)

Windowed Audio:  [[ 0.00000000e+00 -7.87594207e-11  0.00000000e+00 ...  7.08830338e-10
   0.00000000e+00 -7.87594207e-11]
 [ 0.00000000e+00  2.36278262e-10  1.26014777e-09 ... -5.24534450e-08
  -4.72555412e-09 -3.30789567e-09]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -7.23006945e-08
  -6.11171666e-08 -2.52030146e-09]
 ...
 [ 0.00000000e+00  3.93797103e-11  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  3.15036941e-10 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]]


In [75]:
# FFT
audio_winT = np.transpose(audio_win)
audio_fft = np.empty(
    (1 + FFT_SIZE // 2, audio_winT.shape[1]),
    dtype = np.complex64
)

for i in range(audio_fft.shape[1]):
    result = fft.fft(audio_winT[:, i])
    audio_fft[:, i] = result[:audio_fft.shape[0]]

audio_fft = np.transpose(audio_fft)

In [77]:
print("Audio FFT: ", audio_fft)

Audio FFT:  [[-7.8038000e-05-0.0000000e+00j  2.4045346e-04+5.7174724e-20j
  -2.1709736e-06+1.0693791e-19j ... -1.7230058e-04+1.8825307e-19j
  -1.8353053e-04+1.6559494e-19j  3.6294534e-04-0.0000000e+00j]
 [ 3.1598948e-02-0.0000000e+00j  2.7166028e-02+1.5560934e-02j
   1.5942879e-02+2.5217749e-02j ...  1.6111214e-04-3.4005807e-05j
  -1.8181770e-04-4.2044167e-06j  3.6683417e-04-0.0000000e+00j]
 [ 1.3010056e-01-0.0000000e+00j -6.8450995e-02+1.4116479e-01j
  -1.7772962e-01-1.2763330e-01j ... -1.4676573e-04+6.3918778e-05j
  -1.8498139e-04-1.9030572e-06j  3.7119942e-04-0.0000000e+00j]
 ...
 [ 2.5138367e-04-0.0000000e+00j  9.5995703e-05-2.2224674e-04j
  -1.5205175e-04-1.5544654e-04j ...  3.0347674e-05-3.4941637e-05j
  -2.4728393e-05-5.4011070e-05j -6.4191430e-05-0.0000000e+00j]
 [ 5.1429047e-06-0.0000000e+00j  4.5989000e-06-2.2810314e-06j
   3.0845520e-06-4.0712903e-06j ... -1.5424285e-06-2.3229973e-06j
  -2.4648202e-06-1.3224293e-06j -2.7999668e-06-0.0000000e+00j]
 [ 0.0000000e+00-0.0000000e+

In [78]:
fft_power = np.square(audio_fft)
print(fft_power)

[[ 6.0899294e-09+0.0000000e+00j  5.7817868e-08+2.7495722e-23j
   4.7131266e-12-4.6431875e-25j ...  2.9687492e-08-6.4872229e-23j
   3.3683456e-08-6.0783456e-23j  1.3172932e-07-0.0000000e+00j]
 [ 9.9849352e-04-0.0000000e+00j  4.9585040e-04+8.4545754e-04j
  -3.8175948e-04+8.0408703e-04j ...  2.4800727e-08-1.0957496e-08j
   3.3039999e-08+1.5288747e-09j  1.3456730e-07-0.0000000e+00j]
 [ 1.6926156e-02-0.0000000e+00j -1.5241961e-02-1.9325741e-02j
   1.5297558e-02+4.5368437e-02j ...  1.7454569e-08-1.8762172e-08j
   3.4214491e-08+7.0406031e-10j  1.3778902e-07-0.0000000e+00j]
 ...
 [ 6.3193745e-08-0.0000000e+00j -4.0178442e-08-4.2669466e-08j
  -1.0438915e-09+4.7271840e-08j ... -2.9993674e-10-2.1207949e-09j
  -2.3057023e-09+2.6712139e-09j  4.1205399e-09+0.0000000e+00j]
 [ 2.6449469e-11-0.0000000e+00j  1.5946777e-11-2.0980471e-11j
  -7.0609434e-12-2.5116213e-11j ... -3.0172303e-12+7.1661145e-12j
   4.3265196e-12+6.5191008e-12j  7.8398139e-12+0.0000000e+00j]
 [ 0.0000000e+00-0.0000000e+00j  0.00000